# 02 - Prepare the held-out test events

**Settings:** Accelerator = None (CPU), Internet = On. Run with *Save Version > Save & Run All*.

Downloads the 256x256 test chips of Weihui (China), Nova Kakhovka (Ukraine) and Jubba (Somalia, two
image-label pairs) plus their full-frame reference maps, and adds the auxiliary layers.
Note: 997 of the 2,029 published Jubba_2 chips are empty files on the server and are skipped.
Output: `/kaggle/working/ufm-test` (~4 GB).

In [ ]:
# ---- Settings -------------------------------------------------------------
GITHUB_REPO = "https://github.com/mhdnasim6-boop/urban-flood-mapping.git"   # <- edit
BRANCH = "main"
# ---------------------------------------------------------------------------
# Private repo: add a Kaggle secret named GITHUB_TOKEN (Add-ons > Secrets).
# Optional: a secret named HF_TOKEN (Hugging Face read token) avoids download rate limits.
import os, subprocess
def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None
url = GITHUB_REPO
if secret("GITHUB_TOKEN"):
    url = url.replace("https://", f"https://{secret('GITHUB_TOKEN')}@")
if secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = secret("HF_TOKEN")
subprocess.run(["rm", "-rf", "/tmp/ufm"])
subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, "/tmp/ufm"], check=True)
os.chdir("/tmp/ufm")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q rasterio 2>/dev/null; python -c "import rasterio; print('rasterio', rasterio.__version__)"

In [ ]:
!python scripts/prepare_test.py --out /kaggle/working/ufm-test --threads 16

In [ ]:
!python scripts/build_aux.py --root /kaggle/working/ufm-test --cache /tmp/tiles --workers 4 --max-cache-gb 6

In [ ]:
!du -sh /kaggle/working/ufm-test/*
!cat /kaggle/working/ufm-test/event_stats.json